# Bag-of-words embeddings with dhlab

Each text in the subset is tokenized with `nb_tokenizer`, and every token is weighted by
`log(1 + count) * log(max_freq / freq)`, where `count` is its count in the text, `freq` is its total
frequency in the National Library's collection (`dhlab` `totals`) and `max_freq` is the frequency of
the most common token (`.`).
Tokens outside the top 500 000 get the frequency of the 500 000th word. The vocabulary is every
token in the subset.

In [1]:
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from dhlab.api.dhlab_api import totals
from nb_tokenizer import tokenize

SUBSET_DIR = Path("data/maalfrid/subset")
TOP_WORDS = 500_000

## Read and tokenize the subset

In [2]:
paths = sorted(SUBSET_DIR.glob("*/*.txt"))
texts = pd.Series(
    [path.read_text(encoding="utf-8") for path in paths],
    index=pd.Index([f"{path.parent.name}/{path.stem}" for path in paths]),
)
counts = {doc_id: Counter(tokenize(text)) for doc_id, text in texts.items()}
len(counts)

350

## Reference frequencies

In [3]:
freqs = totals(TOP_WORDS)["freq"]
min_freq = freqs.iloc[-1]
max_freq = freqs.iloc[0]
freqs.head(), min_freq, max_freq

(.     7655423257
 ,     5052171514
 i     2531262027
 og    2520268056
 -     1314451583
 Name: freq, dtype: int64,
 np.int64(4625),
 np.int64(7655423257))

## Bag-of-words matrix

Rows are documents, columns are every token in the subset. Log-scaling the counts keeps
frequent tokens from dominating, and `log(max_freq / freq)` gives the most common token (`.`) weight
0 and rare tokens the most weight.

In [4]:
bow_counts = pd.DataFrame.from_dict(counts, orient="index").fillna(0)
bow_counts = bow_counts.reindex(sorted(bow_counts.columns), axis=1)

scaling = freqs.reindex(bow_counts.columns).fillna(min_freq)
bow = np.log1p(bow_counts) * np.log(max_freq / scaling)

num_docs, vocab_size = bow.shape
print(
    f"Number of documents: {num_docs}\nNumber of unique tokens across documents: {vocab_size}"
)

Number of documents: 350
Number of unique tokens across documents: 29745


In [5]:
print(
    f"{scaling.eq(min_freq).mean():.1%} of the vocabulary is outside the top {TOP_WORDS:,} words"
)
bow.iloc[:5, :10]

30.8% of the vocabulary is outside the top 500,000 words


,!,"""",#,%,&,',(,),*,+
22julisenteret.no/html_0ddebc78-75b6-4cf0-b2d2-acc4300ea741,0.00000,0.0,0.0,0.0,0.0,0.0,1.597275,1.512794,0.0,0.0
22julisenteret.no/html_3f77b1b4-1b00-439c-a493-f2f8a35a443d,0.00000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0
22julisenteret.no/html_628f3a16-0439-43bf-981b-d47ca6723547,2.24032,0.0,0.0,0.0,0.0,0.0,4.484117,4.246950,0.0,0.0
22julisenteret.no/html_872d59ed-7be6-4403-b552-ce477520079a,0.00000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0
22julisenteret.no/html_9992bd8e-b5f8-4df2-a252-7fcad24f5d52,0.00000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0


## 2D projection

Flattened with `visualize.project_2d` (L2-normalise, truncated SVD to 50 dimensions, t-SNE),
shared with the other embedding notebooks.

In [6]:
from visualize import plot_by_domain, project_2d

projection = project_2d(bow.to_numpy(), bow.index)
projection.head()

,x,y,domain
22julisenteret.no/html_0ddebc78-75b6-4cf0-b2d2-acc4300ea741,-10.791012,-6.100647,22julisenteret.no
22julisenteret.no/html_3f77b1b4-1b00-439c-a493-f2f8a35a443d,-12.211635,-6.328246,22julisenteret.no
22julisenteret.no/html_628f3a16-0439-43bf-981b-d47ca6723547,-1.701250,-6.548666,22julisenteret.no
22julisenteret.no/html_872d59ed-7be6-4403-b552-ce477520079a,-13.323604,-7.612084,22julisenteret.no
22julisenteret.no/html_9992bd8e-b5f8-4df2-a252-7fcad24f5d52,-11.700929,-5.258424,22julisenteret.no


## Scatter plot by domain

Pick a domain in the dropdown to highlight it against all the other documents in grey.
Hover a point to see its domain, and the first 1000 characters of its text in the panel to the
right.

In [7]:
plot_by_domain(projection, texts, "t-SNE of BOW embeddings")

## Most and least similar domains

Each domain is represented by its centroid (the mean of its document embeddings), and domains
are compared by the cosine similarity of their centroids.

In [8]:
from embedding_similarity import (
    centroid_similarities,
    compute_centroids,
    most_and_least_similar,
)

centroids = compute_centroids(bow.to_numpy(), projection["domain"])
similarities = centroid_similarities(centroids)
most_and_least_similar(similarities).round(3)

,most_similar,most_similarity,least_similar,least_similarity
22julisenteret.no,forskning.no,0.374,digitalarkivet.no,0.110
aho.no,forskning.no,0.293,digitalarkivet.no,0.084
aldringoghelse.no,husbanken.no,0.333,digitalarkivet.no,0.088
arbeidsretten.no,bioteknologiradet.no,0.228,varsom.no,0.068
arbeidstilsynet.no,helsenorge.no,0.400,digitalarkivet.no,0.105
artsdatabanken.no,forskning.no,0.240,digitalarkivet.no,0.060
banenor.no,arbeidstilsynet.no,0.259,varsom.no,0.092
barneombudet.no,bibliotekutvikling.no,0.300,digitalarkivet.no,0.078
bibliotekutvikling.no,husbanken.no,0.421,digitalarkivet.no,0.111
bioteknologiradet.no,forskning.no,0.418,digitalarkivet.no,0.098
